# Lumis 18: add-ons on top of the MedSigLIP model

Notebook 16 chose **MedSigLIP + extra cases** (logistic regression on MedSigLIP embeddings): 76% accuracy and 67%
average sensitivity on SCIN phone photos, against 37% and 35% for v1. This notebook tests five add-ons on top of it,
each on its own, on the **same 482 cases and the same 5 folds**:

| | Add-on | What changes |
|---|---|---|
| A | Wider strength range | C chosen from 0.001 to 0.03 (notebook 16 always chose 0.01, the edge of its range) |
| B | All photos of a case | the case's photos (up to 3) are scored and their results averaged |
| C | Small neural network head | one hidden layer (256), instead of logistic regression; fixed settings, no tuning |
| D | Intake questions | the person's answers (body part, symptoms, texture, duration, age group, sex) added as inputs; DermNet photos and skipped questions count as blank |
| E | Claude's answer | Claude Haiku's own pick (notebook 16) combined with the model by Bayes' rule, using how often Claude is right per condition in the other folds |

**Rules fixed 2026-11-13, before this notebook ran:**
1. An add-on is adopted only if, against the base model on the same cases: the **95% interval** (paired bootstrap)
   of its gain in average sensitivity is **above zero**, **tinea** sensitivity is **not lower**, and **scabies**
   (6 cases) does **not lose a case**.
2. For D (questions are optional in the app): with **every answer left blank**, average sensitivity must not fall more
   than **2 points** below the base model.
3. Adopted add-ons are then combined (head: C if adopted, otherwise logistic regression with A's range if adopted;
   plus D, B and E if adopted). The combination is used only if its average sensitivity is at least that of the best
   single adopted add-on (and, if D is in it, it also passes rule 2 with every answer skipped); otherwise that
   single add-on is used.
4. The SCIN test split is not used here. Notebook 19 fits the chosen model, calibrates it and tests it once.

Disclosed: the 482 cases are reused from notebook 16, so these numbers are optimistic; only notebook 19's test is not.

**Runtime: CPU is fine**, about 30 to 45 minutes. No secrets needed (Claude's answers from notebook 16 are reused).

In [ ]:
SEED = 42
N_FOLDS = 5
PCA_DIM = 256
C_BASE = [0.01, 0.1, 1.0]          # notebook 16
C_WIDE = [0.001, 0.003, 0.01, 0.03]  # add-on A
SCIN_W = 3.0
MAX_WEIGHT = 5.0
MIN_EXTRA_AGREEMENT = 0.25
BLANK_MAX_DROP = 0.02
N_BOOT = 2000
ZERO_SHOT_MODEL = 'claude-haiku-5-5'
FLAG_PREFIXES = ('body_parts_', 'condition_symptoms_', 'other_symptoms_', 'textures_')
CATEGORICAL = ['age_group', 'sex_at_birth', 'condition_duration']

In [ ]:
!pip install -q google-cloud-storage
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, io, json, time, zipfile, hashlib
import numpy as np, pandas as pd

OUT = f'{BASE}/exports/addons'
os.makedirs(OUT, exist_ok=True)
if os.path.exists(f'{OUT}/decision.json'):
    raise RuntimeError(f'{OUT}/decision.json exists: the add-on round was already decided. Do not rerun it to try again.')
dec16 = json.load(open(f'{BASE}/exports/foundation/decision.json'))
assert dec16['winner'] == 'medsiglip + extra', f'Notebook 16 chose {dec16["winner"]}; this notebook is written for MedSigLIP + extra.'

for z, marker, dest in [(f'{BASE}/data/dermnet.zip', '/content/dermnet', '/content/dermnet'),
                        (f'{BASE}/data/scin_images.zip', '/content/scin', '/content'),
                        (f'{BASE}/data/scin_extra_images.zip', '/content/scin_extra', '/content')]:
    if not os.path.exists(marker):
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
V1_RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
cal = json.load(open(f'{BASE}/exports/runs/{V1_RUN}/calibration.json'))
CLASSES = cal['classes']
K = len(CLASSES)
IDX = {c: i for i, c in enumerate(CLASSES)}
ADJ = np.array(cal['logit_adjust'])

derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
d_train = derm[derm.split == 'train'].reset_index(drop=True)
d_val = derm[derm.split == 'val'].reset_index(drop=True)
d_train['file'] = '/content/dermnet/' + d_train.path
d_val['file'] = '/content/dermnet/' + d_val.path
split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
tv = split[split.split.isin(['train', 'val'])].reset_index(drop=True)
tv['file'] = '/content/' + tv.path
extra = pd.read_csv(f'{BASE}/data/scin_extra_v1.csv', dtype={'case_id': str})
extra['file'] = '/content/' + extra.path
print(f'DermNet train {len(d_train)}, val {len(d_val)} | SCIN train+val {len(tv)} photos | extra {len(extra)} photos')

## 1. SCIN answers to the intake questions

In [ ]:
import os, io, ast, json, time, hashlib, zipfile
import numpy as np, pandas as pd
from google.cloud import storage

BUCKET = 'dx-scin-public-data'
try:
    bucket = storage.Client.create_anonymous_client().bucket(BUCKET)
    bucket.blob('dataset/scin_cases.csv').download_as_bytes(start=0, end=10)
except Exception as e:
    print('Anonymous access failed, signing in instead (a popup will ask you to allow access):', type(e).__name__)
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project='dx-scin-public').bucket(BUCKET)

def fetch(path):
    return bucket.blob(path).download_as_bytes()

cases = pd.read_csv(io.BytesIO(fetch('dataset/scin_cases.csv')), dtype={'case_id': str})
labels = pd.read_csv(io.BytesIO(fetch('dataset/scin_labels.csv')), dtype={'case_id': str})
scin = cases.merge(labels, on='case_id')
print(len(scin), 'SCIN cases')

In [ ]:
def parse(x):
    if isinstance(x, str) and x.strip():
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

def to_lumis(name):
    n = name.lower()
    if 'scabies' in n:
        return 'scabies'
    if 'psoria' in n and 'parapsoria' not in n:
        return 'psoriasis'
    if 'tinea' in n or 'dermatophyt' in n or 'ringworm' in n:
        return 'tinea'
    if n.startswith('acne') and 'rosacea' not in n and 'keloid' not in n:
        return 'acne'
    if ('eczema' in n or 'atopic' in n) and 'contact' not in n:
        return 'eczema'
    return None

def main_label(w):
    if not w:
        return None, 0.0
    name, weight = max(w.items(), key=lambda kv: kv[1])
    return to_lumis(name), weight

def fitz(r):
    vals = [r.get(f'dermatologist_fitzpatrick_skin_type_label_{i}') for i in (1, 2, 3)]
    vals = [v for v in vals if isinstance(v, str) and v.startswith('FST')]
    if not vals:
        return None
    v = int(round(np.median([int(x[3]) for x in vals])))
    return 'I-II' if v <= 2 else ('III-IV' if v <= 4 else 'V-VI')

scin['weights'] = scin.weighted_skin_condition_label.map(parse)
scin[['lumis_class', 'agreement']] = scin.weights.apply(lambda w: pd.Series(main_label(w)))
scin['tone'] = scin.apply(fitz, axis=1)

def yes(col):
    return col.map(lambda v: 1.0 if (v is True or v == 1 or (isinstance(v, str) and v.strip().upper() in ('YES', 'TRUE', '1'))) else 0.0)

flag_cols = [c for c in scin.columns if c.startswith(FLAG_PREFIXES)]
cat_cols = [c for c in CATEGORICAL if c in scin.columns]
assert flag_cols, 'No question columns found in scin_cases.csv.'
Q = pd.concat([pd.DataFrame({c: yes(scin[c]) for c in flag_cols}),
               pd.get_dummies(scin[cat_cols].fillna('MISSING').astype(str), prefix=cat_cols).astype(float)], axis=1)
Q.index = scin.case_id.values
Q = Q[~Q.index.duplicated()]
BLANK = pd.Series(0.0, index=Q.columns)
for c in cat_cols:
    if f'{c}_MISSING' in BLANK.index:
        BLANK[f'{c}_MISSING'] = 1.0
print(f'{Q.shape[1]} question inputs; a fully skipped form = all yes/no answers 0 and every choice "MISSING"')

## 2. Same photos, embeddings and folds as notebook 16

In [ ]:
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore', message='The least populated class')

def readable(files):
    ok = []
    for f in files:
        try:
            Image.open(f).verify()
            ok.append(f)
        except Exception:
            print('cannot be read, left out:', f)
    return set(ok)

good = readable(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file))
d_train, d_val = d_train[d_train.file.isin(good)].reset_index(drop=True), d_val[d_val.file.isin(good)].reset_index(drop=True)
extra = extra[extra.file.isin(good)].reset_index(drop=True)
tv = tv[tv.file.isin(good)].reset_index(drop=True)

ALL = list(dict.fromkeys(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file)))
KEY = hashlib.md5('\n'.join(ALL).encode()).hexdigest()[:10]
d = f'{BASE}/exports/embeddings/medsiglip'
n_chunks = (len(ALL) + 255) // 256
missing = [k for k in range(n_chunks) if not os.path.exists(f'{d}/{KEY}_{k:04d}.npy')]
assert not missing, f'MedSigLIP embeddings for this photo list are missing ({len(missing)} chunks): run notebook 16 section 3 first.'
E = np.concatenate([np.load(f'{d}/{KEY}_{k:04d}.npy') for k in range(n_chunks)])
row = {f: i for i, f in enumerate(ALL)}
print('MedSigLIP embeddings', E.shape)

parent = {c: c for c in tv.case_id.unique()}
def find(c):
    while parent[c] != c:
        parent[c] = parent[parent[c]]
        c = parent[c]
    return c
for _, g in tv.groupby('md5'):
    cs = g.case_id.unique()
    for c in cs[1:]:
        parent[find(c)] = find(cs[0])
tv['group'] = tv.case_id.map(find)
ev = tv[tv.eval_photo].reset_index(drop=True)
y_ev = ev.lumis_class.map(IDX).values
fold_of = {}
for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(ev, y_ev, ev.group)):
    for c in ev.case_id.iloc[te]:
        fold_of[c] = f
tv['fold'] = tv.case_id.map(fold_of)
ev['fold'] = ev.case_id.map(fold_of)
print(pd.crosstab(ev.lumis_class, ev.fold, margins=True))

## 3. Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch

def balanced(labels, cap=None):
    labels = np.asarray(labels)
    w = len(labels) / (K * np.maximum(np.bincount(labels, minlength=K), 1))
    return (np.minimum(w, cap) if cap else w)[labels]

def macro_sens(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in range(K) if (y == k).any()]))

def scores(y, p):
    s = {'accuracy': float((p == y).mean()), 'macro_sensitivity': macro_sens(y, p)}
    for k, c in enumerate(CLASSES):
        s[f'sens_{c}'] = float((p[y == k] == k).mean()) if (y == k).any() else np.nan
    s['scabies_correct'] = int(((p == IDX['scabies']) & (y == IDX['scabies'])).sum())
    return s

def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))

yd = d_train.lumis_class.map(IDX).values

def qrows(case_ids, blank=False):
    """Question inputs for SCIN photos (by case); DermNet photos and blank=True get the skipped-form vector."""
    if blank:
        return np.tile(BLANK.values, (len(case_ids), 1))
    return np.stack([Q.loc[c].values if c in Q.index else BLANK.values for c in case_ids])

def training_set(scin_rows, use_q):
    pool = pd.concat([scin_rows.assign(mult=1.0), extra.assign(mult=extra.agreement.astype(float))])
    yp = pool.lumis_class.map(IDX).values.astype(int)
    files = list(d_train.file) + list(pool.file)
    X = E[[row[f] for f in files]]
    Xq = np.concatenate([qrows([None] * len(d_train), blank=True), qrows(list(pool.case_id))]) if use_q else None
    y = np.concatenate([yd, yp])
    w = np.concatenate([balanced(yd), SCIN_W * balanced(yp, MAX_WEIGHT) * pool.mult.values])
    return X, Xq, y, w

class Head:
    """Standardize -> PCA(256) on the image part; question inputs appended; then LR or a small neural network."""
    def __init__(self, kind, C=None):
        self.kind, self.C = kind, C

    def fit(self, X, Xq, y, w, prep=None):
        if prep is None:
            sc = StandardScaler().fit(X)
            prep = (sc, PCA(PCA_DIM, random_state=SEED).fit(sc.transform(X)))
        self.prep = prep
        Z = self._z(X, Xq)
        if self.kind == 'lr':
            self.m = LogisticRegression(C=self.C, max_iter=3000).fit(Z, y, sample_weight=w)
        else:
            torch.manual_seed(SEED)
            self.m = torch.nn.Sequential(torch.nn.Linear(Z.shape[1], 256), torch.nn.ReLU(), torch.nn.Dropout(0.3),
                                         torch.nn.Linear(256, K))
            opt = torch.optim.AdamW(self.m.parameters(), lr=1e-3, weight_decay=1e-2)
            Zt, yt, wt = torch.tensor(Z, dtype=torch.float32), torch.tensor(y), torch.tensor(w / w.mean(), dtype=torch.float32)
            g = torch.Generator().manual_seed(SEED)
            self.m.train()
            for _ in range(60):
                for idx in torch.randperm(len(Zt), generator=g).split(256):
                    loss = (torch.nn.functional.cross_entropy(self.m(Zt[idx]), yt[idx], reduction='none') * wt[idx]).mean()
                    opt.zero_grad(); loss.backward(); opt.step()
            self.m.eval()
        return self

    def _z(self, X, Xq):
        Z = self.prep[1].transform(self.prep[0].transform(X))
        return np.concatenate([Z, Xq], 1) if Xq is not None else Z

    def logits(self, X, Xq):
        Z = self._z(X, Xq)
        if self.kind == 'lr':
            return self.m.decision_function(Z) + ADJ
        with torch.no_grad():
            return self.m(torch.tensor(Z, dtype=torch.float32)).numpy() + ADJ

In [ ]:
def choose_C(tr, use_q, grid, prep):
    tr_ev = tr[tr.eval_photo].reset_index(drop=True)
    sc = {C: [] for C in grid}
    for _, iv in StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(tr_ev, tr_ev.lumis_class, tr_ev.group):
        held = set(tr_ev.case_id.iloc[iv])
        X, Xq, y, w = training_set(tr[~tr.case_id.isin(held)], use_q)
        vs = tr_ev.iloc[iv]
        Xv = E[[row[f] for f in vs.file]]
        Xvq = qrows(list(vs.case_id)) if use_q else None
        yv = vs.lumis_class.map(IDX).values
        for C in grid:
            sc[C].append(macro_sens(yv, Head('lr', C).fit(X, Xq, y, w, prep).logits(Xv, Xvq).argmax(1)))
    return max(grid, key=lambda C: (np.mean(sc[C]), -C))

def case_logits(model, cases, use_q, multi, blank=False):
    """Log-probabilities per case: the scored close-up, or the mean over all of the case's photos."""
    out = []
    for c in cases:
        rows = tv[tv.case_id == c] if multi else ev[ev.case_id == c]
        X = E[[row[f] for f in rows.file]]
        Xq = qrows(list(rows.case_id), blank=blank) if use_q else None
        out.append(log_softmax(model.logits(X, Xq)).mean(0))
    return np.stack(out)

def run(name, head='lr', grid=C_BASE, use_q=False, multi=False):
    t0 = time.time()
    oof, oof_blank, cs = np.zeros((len(ev), K)), np.zeros((len(ev), K)), []
    for f in range(N_FOLDS):
        tr = tv[tv.fold != f]
        X, Xq, y, w = training_set(tr, use_q)
        sc = StandardScaler().fit(X)
        prep = (sc, PCA(PCA_DIM, random_state=SEED).fit(sc.transform(X)))
        C = choose_C(tr, use_q, grid, prep) if head == 'lr' else None
        cs.append(C)
        model = Head(head, C).fit(X, Xq, y, w, prep)
        te = np.where(ev.fold.values == f)[0]
        oof[te] = case_logits(model, ev.case_id.iloc[te], use_q, multi)
        if use_q:
            oof_blank[te] = case_logits(model, ev.case_id.iloc[te], use_q, multi, blank=True)
    print(f'{name}: C per fold {cs} | {time.time() - t0:.0f} s')
    return oof, (oof_blank if use_q else None)

# Add-on E: Claude's pick, combined by Bayes' rule with a per-fold confusion matrix (from the other folds)
zs = {}
for line in open(f'{BASE}/exports/zero_shot/answers_{ZERO_SHOT_MODEL}.jsonl'):
    r = json.loads(line)
    if r.get('ok'):
        zs[r['file']] = IDX[r['condition']]
claude = np.array([zs.get(f, -1) for f in ev.file])
print(f'Claude answers available for {(claude >= 0).sum()} of {len(ev)} cases')

def with_claude(oof):
    out = oof.copy()
    for f in range(N_FOLDS):
        other, te = ev.fold.values != f, np.where(ev.fold.values == f)[0]
        M = np.ones((K, K))
        for t, c in zip(y_ev[other], claude[other]):
            if c >= 0:
                M[t, c] += 1
        M = M / M.sum(1, keepdims=True)                      # P(Claude says j | true condition k)
        for i in te:
            if claude[i] >= 0:
                out[i] = out[i] + np.log(M[:, claude[i]])
    return out

## 4. Base and add-ons (rules 1 and 2)

In [ ]:
oofs, blanks = {}, {}
oofs['base'], _ = run('base (notebook 16 winner)')
print(f'  check: notebook 16 had average sensitivity {dec16["results"]["medsiglip + extra"]["macro_sensitivity"]:.4f}, '
      f'now {macro_sens(y_ev, oofs["base"].argmax(1)):.4f}')
oofs['A wider C'], _ = run('A wider C', grid=C_WIDE)
oofs['B all photos'], _ = run('B all photos', multi=True)
oofs['C neural head'], _ = run('C neural head', head='mlp')
oofs['D questions'], blanks['D questions'] = run('D questions', use_q=True)
oofs['E + Claude'] = with_claude(oofs['base'])

In [ ]:
rng = np.random.default_rng(SEED)
boots = [rng.integers(0, len(ev), len(ev)) for _ in range(N_BOOT)]
def gain_ci(p1, p0):
    d = [macro_sens(y_ev[b], p1[b]) - macro_sens(y_ev[b], p0[b]) for b in boots]
    return [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]

p_base = oofs['base'].argmax(1)
s_base = scores(y_ev, p_base)
results, checks, adopted = {'base': s_base}, {}, []
for name in ['A wider C', 'B all photos', 'C neural head', 'D questions', 'E + Claude']:
    p = oofs[name].argmax(1)
    s = scores(y_ev, p)
    ci = gain_ci(p, p_base)
    c = {'gain interval above zero': ci[0] > 0,
         'tinea not lower': s['sens_tinea'] >= s_base['sens_tinea'],
         'scabies keeps every case': s['scabies_correct'] >= s_base['scabies_correct']}
    if name == 'D questions':
        sb = scores(y_ev, blanks[name].argmax(1))
        results['D questions, all answers skipped'] = sb
        c['all answers skipped: at most 2 points below base'] = sb['macro_sensitivity'] >= s_base['macro_sensitivity'] - BLANK_MAX_DROP
    results[name] = s
    checks[name] = {**c, 'gain': s['macro_sensitivity'] - s_base['macro_sensitivity'], 'gain_ci95': ci}
    print(f'\n{name}: average sensitivity {s["macro_sensitivity"]:.3f} (gain {checks[name]["gain"]:+.3f}, '
          f'95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]), accuracy {s["accuracy"]:.3f}')
    for k, ok in c.items():
        print(f'   {"ok  " if ok else "FAIL"}  {k}')
    if all(c.values()):
        adopted.append(name)
print('\nAdopted add-ons:', adopted or 'none')

## 5. Combination (rule 3)

In [ ]:
final = 'base'
if adopted:
    best_single = max(adopted, key=lambda n: results[n]['macro_sensitivity'])
    final = best_single
    if len(adopted) > 1:
        head = 'mlp' if 'C neural head' in adopted else 'lr'
        grid = C_WIDE if 'A wider C' in adopted else C_BASE
        comb, comb_blank = run('combination', head=head, grid=grid, use_q='D questions' in adopted, multi='B all photos' in adopted)
        if 'E + Claude' in adopted:
            comb = with_claude(comb)
            comb_blank = with_claude(comb_blank) if comb_blank is not None else None
        oofs['combination'] = comb
        results['combination'] = scores(y_ev, comb.argmax(1))
        if comb_blank is not None:
            results['combination, all answers skipped'] = scores(y_ev, comb_blank.argmax(1))
        blank_ok = comb_blank is None or results['combination, all answers skipped']['macro_sensitivity'] >= s_base['macro_sensitivity'] - BLANK_MAX_DROP
        if results['combination']['macro_sensitivity'] >= results[best_single]['macro_sensitivity'] and blank_ok:
            final = 'combination'
print('FINAL CHOICE:', final, ('= ' + ' + '.join(adopted)) if final == 'combination' else '')

table = pd.DataFrame(results).T
display(table[['accuracy', 'macro_sensitivity'] + [f'sens_{c}' for c in CLASSES] + ['scabies_correct']].round(3))

tone = ev.tone.fillna('unknown').values
display(pd.DataFrame({n: {t: round(float((oofs[n].argmax(1)[tone == t] == y_ev[tone == t]).mean()), 3) for t in sorted(set(tone))}
                      for n in dict.fromkeys(['base', final])} | {'cases': {t: int((tone == t).sum()) for t in sorted(set(tone))}}))

In [ ]:
table.to_csv(f'{OUT}/cv_results.csv')
np.savez(f'{OUT}/oof_logits.npz', y=y_ev, **{n.replace(' ', '_').replace('+', 'plus'): o for n, o in oofs.items()})
json.dump({'final': final, 'adopted': adopted, 'checks': checks,
           'results': {k: {m: (None if pd.isna(v) else v) for m, v in r.items()} for k, r in results.items()},
           'question_columns': list(Q.columns), 'blank_vector': BLANK.to_dict(),
           'settings': {'c_base': C_BASE, 'c_wide': C_WIDE, 'pca_dim': PCA_DIM, 'scin_w': SCIN_W, 'max_weight': MAX_WEIGHT,
                        'mlp': '1 hidden layer 256, ReLU, dropout 0.3, AdamW lr 1e-3 wd 1e-2, 60 epochs, batch 256',
                        'blank_max_drop': BLANK_MAX_DROP, 'zero_shot_model': ZERO_SHOT_MODEL},
           'created_at': time.strftime('%Y-%m-%d %H:%M')},
          open(f'{OUT}/decision.json', 'w'), indent=1, default=float)
print('Saved', f'{OUT}/decision.json')